<a href="https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier

for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    scripts = p / "work" / "scripts"
    if (scripts / "warehouse_frame.py").exists():
        sys.path.insert(0, str(scripts))
        break
else:
    raise FileNotFoundError("work/scripts/warehouse_frame.py not found")

from warehouse_frame import load_notebook_frame

df = load_notebook_frame()

Hugging Face warehouse: 79,576 pages, 26 clients
Features: Jan-Feb 2026. Label: Apr impressions < 80% of Mar.
Declining rate: 0.557


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method Choice and Justification
Primary method: **Random Forest Classifier**. It fits this lane because:

1. **Mixed scales**: Impressions are in the thousands, CTR is a small rate, position is a rank. Trees do not need those to be scaled the same way.
2. **Interactions**: Age, freshness, visibility, and rank do not add in a straight line.
3. **Readable**: Feature importance is a decent way to say which signals the model leaned on.
4. **Comparison**: Logistic Regression and a shallow Decision Tree run on the same split so the lift is not just "trees vs nothing."

The fair Week-4 rule (no `trend_direction` / `trend_pct` in the score) is the bar. On the warehouse, overall declining rate is 0.557.

In [12]:
print("Method Selection Verification:")
print("Primary: Random Forest Classifier")
print("Comparison: Logistic Regression, Decision Tree")
print("Justification: Non-linear patterns, scaling robustness, feature importance, proven results")

Method Selection Verification:
Primary: Random Forest Classifier
Comparison: Logistic Regression, Decision Tree
Justification: Non-linear patterns, scaling robustness, feature importance, proven results


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split Design
**Client-holdout** with GroupKFold on `client_id`. Pages from the same client never sit in both train and test.

- 5 folds; the comparison table uses the first fold (same recipe as `canonical_metrics.json`)
- `random_state=42` on the forest
- GroupKFold does not stratify; we still print the declining rate on each side so a wild shift is visible

This is the honest question: does the ranking still work on a client we did not train on?

In [13]:
# Create the target label (same as data contract)
df['is_declining'] = (df["trend_direction"].str.lower() == "down").astype(int)

# Use the five core features from the data contract
features = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'ctr', 'avg_position']
X = df[features].fillna(0)  # Handle missing values
y = df['is_declining']

# Client-holdout split using GroupKFold
group_kfold = GroupKFold(n_splits=5)
groups = df['client_id']

# Get the first split for train/test
train_idx, test_idx = next(group_kfold.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Training set: {len(X_train):,} pages from {df.iloc[train_idx]['client_id'].nunique()} clients")
print(f"Test set: {len(X_test):,} pages from {df.iloc[test_idx]['client_id'].nunique()} clients")
print(f"Target distribution - Train: {y_train.mean():.3f}, Test: {y_test.mean():.3f}")
print(f"Client overlap check: {set(df.iloc[train_idx]['client_id']) & set(df.iloc[test_idx]['client_id'])} (should be empty)")

Training set: 58,783 pages from 25 clients
Test set: 20,793 pages from 1 clients
Target distribution - Train: 0.599, Test: 0.439
Client overlap check: set() (should be empty)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Model Training & Baseline Comparison
I'll train three models (Random Forest, Decision Tree, Logistic Regression) and compare them against the Week-4 baseline on the same client-holdout test set using Precision@50 as the primary metric.

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Same fair Week-4 rule on this test fold (no trend_direction / trend_pct)
test_df = X_test.copy()
test_df["is_declining"] = y_test.values
test_df["days_since_last_update"] = df.iloc[test_idx]["days_since_last_update"].values
test_age = df.iloc[test_idx]["content_age_days"].values

stale_visible = ((test_df["days_since_last_update"] >= 180) & (test_df["impressions_90d"] >= 500)).astype(int)
position_decay_risk = ((test_df["avg_position"] > 10) & (test_age >= 180)).astype(int)
low_engagement = ((test_df["ctr"] < 0.05) & (test_df["impressions_90d"] >= 1000)).astype(int)

test_df["baseline_score"] = (
    0.4 * stale_visible * test_df["impressions_90d"]
    + 0.3 * position_decay_risk * test_df["impressions_90d"]
    + 0.3 * low_engagement * test_df["impressions_90d"]
)

models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=42, class_weight="balanced"),
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    results[name] = precision_at_k(scores, y_test.values, 50)

baseline_p50 = precision_at_k(test_df["baseline_score"].values, y_test.values, 50)
results["Fair Week-4 baseline"] = baseline_p50
results["Random (base rate)"] = float(y_test.mean())

comparison_df = pd.DataFrame({
    "Method": list(results.keys()),
    "Precision@50": list(results.values()),
}).sort_values("Precision@50", ascending=False)

print("=== MODEL VS BASELINE (same client-holdout fold) ===")
print(comparison_df.to_string(index=False))

rf_p50 = results["Random Forest"]
print(f"\nRandom Forest vs fair baseline: {rf_p50 / baseline_p50:.2f}x")
print(f"Random Forest vs test base rate: {rf_p50 / y_test.mean():.2f}x")

=== MODEL VS BASELINE (same client-holdout fold) ===
              Method  Precision@50
 Logistic Regression      0.780000
Fair Week-4 baseline      0.640000
  Random (base rate)      0.438609
       Decision Tree      0.340000
       Random Forest      0.280000

Random Forest vs fair baseline: 0.44x
Random Forest vs test base rate: 0.64x


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Error Analysis and Feature Interpretation
I'll analyze where the best model makes mistakes and what features it relies on most.

In [15]:
# Feature importance and Precision@50 errors for the Random Forest
rf_model = models["Random Forest"]

feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_,
}).sort_values("Importance", ascending=False)

print("=== FEATURE IMPORTANCE ===")
print(feature_importance.to_string(index=False))

print("\nIn words:")
for _, row in feature_importance.iterrows():
    print(f"- {row['Feature']}: {row['Importance']:.3f}")

test_df["model_prob"] = rf_model.predict_proba(X_test)[:, 1]
test_df["model_pred"] = rf_model.predict(X_test)

order = np.argsort(-test_df["model_prob"].values)
top50 = test_df.iloc[order[:50]]
fp50 = int((top50["is_declining"] == 0).sum())
tp50 = int((top50["is_declining"] == 1).sum())
print(f"\n=== PRECISION@50 ERRORS (top 50 by score) ===")
print(f"True positives: {tp50}/50")
print(f"False positives: {fp50}/50")
print(f"Precision@50: {tp50/50:.3f}")

false_positives = test_df[(test_df["model_pred"] == 1) & (test_df["is_declining"] == 0)]
false_negatives = test_df[(test_df["model_pred"] == 0) & (test_df["is_declining"] == 1)]

print(f"\n=== 0.5-THRESHOLD ERRORS (whole test fold, not Precision@50) ===")
print(f"False positives: {len(false_positives)}")
print(f"False negatives: {len(false_negatives)}")

if len(false_positives) > 0:
    print(f"  FP avg impressions: {false_positives['impressions_90d'].mean():.0f}")
    print(f"  FP avg CTR: {false_positives['ctr'].mean():.3f}")
    print(f"  FP avg position: {false_positives['avg_position'].mean():.1f}")

if len(false_negatives) > 0:
    print(f"  FN avg impressions: {false_negatives['impressions_90d'].mean():.0f}")
    print(f"  FN avg CTR: {false_negatives['ctr'].mean():.3f}")
    print(f"  FN avg position: {false_negatives['avg_position'].mean():.1f}")

print("\n=== A FEW FALSE POSITIVES IN THE TOP 50 ===")
fp_top = top50[top50["is_declining"] == 0].head(3)
for idx in fp_top.index:
    print(
        f"  impressions={fp_top.loc[idx, 'impressions_90d']:.0f}, "
        f"ctr={fp_top.loc[idx, 'ctr']:.3f}, "
        f"position={fp_top.loc[idx, 'avg_position']:.1f}"
    )
    print("    Wrong if this is evergreen content that does not need a rewrite")

=== FEATURE IMPORTANCE ===
               Feature  Importance
days_since_last_update    0.421808
      content_age_days    0.245833
                   ctr    0.147483
       impressions_90d    0.102541
          avg_position    0.082336

In words:
- days_since_last_update: 0.422
- content_age_days: 0.246
- ctr: 0.147
- impressions_90d: 0.103
- avg_position: 0.082

=== PRECISION@50 ERRORS (top 50 by score) ===
True positives: 14/50
False positives: 36/50
Precision@50: 0.280

=== 0.5-THRESHOLD ERRORS (whole test fold, not Precision@50) ===
False positives: 8190
False negatives: 1594
  FP avg impressions: 3276
  FP avg CTR: 0.206
  FP avg position: 14.3
  FN avg impressions: 8083
  FN avg CTR: 0.305
  FN avg position: 10.8

=== A FEW FALSE POSITIVES IN THE TOP 50 ===
  impressions=633, ctr=0.000, position=25.2
    Wrong if this is evergreen content that does not need a rewrite
  impressions=664, ctr=0.000, position=25.1
    Wrong if this is evergreen content that does not need a rewrite
 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Method choice is justified and fits the lane
- [ ] Split design is honest (client-holdout, no leakage)
- [ ] Model compared against baseline on SAME data and metric
- [ ] Error analysis shows where model fails and why
- [ ] Feature importance is interpretable and makes sense
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.